In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from gensim.models import Word2Vec
from sklearn.ensemble import RandomForestClassifier


dataset_path = '../Email Dataset/Clean_Emails_Dataset.csv'
df = pd.read_csv(dataset_path)


df = df.dropna(subset=['Email Text', 'Email Type'])


df['label_num'] = df['Email Type'].map({'Safe Email': 0, 'Phishing Email': 1})

# 3. Save the processed version for future use
output_path = '../Email Dataset/Processed_Emails_Dataset.csv'
df.to_csv(output_path, index=False)
print(f"Dataset Loaded and Processed. Saved to: {output_path}")

# Define features and labels
X = df['Email Text'].astype(str)
y = df['label_num']

# --- Cell 3: Split the data ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define target names for the reports
target_names = ['Safe Email', 'Phishing Email']

# --- Cell 4 & 5: TF-IDF Approaches ---
print("\n--- Starting TF-IDF Models ---")
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Logistic Regression
lr_model = LogisticRegression()
lr_model.fit(X_train_tfidf, y_train)
y_pred_lr = lr_model.predict(X_test_tfidf)
print("\nLogistic Regression Report:")
print(classification_report(y_test, y_pred_lr, target_names=target_names))

# Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
y_pred_nb = nb_model.predict(X_test_tfidf)
print("\nNaive Bayes Report:")
print(classification_report(y_test, y_pred_nb, target_names=target_names))

# --- Cell 6: Word Embedding Approach (Word2Vec) ---
print("\n--- Starting Word2Vec Model ---")

# Tokenize text for Word2Vec
X_train_tokens = [text.split() for text in X_train]
X_test_tokens = [text.split() for text in X_test]

# Train Word2Vec model
w2v_model = Word2Vec(sentences=X_train_tokens, vector_size=100, window=5, min_count=1, workers=4)

# Function to get average vector
def get_avg_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)

X_train_w2v = np.array([get_avg_vector(tokens, w2v_model) for tokens in X_train_tokens])
X_test_w2v = np.array([get_avg_vector(tokens, w2v_model) for tokens in X_test_tokens])

# Use RandomForest as classifier for Word2Vec
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_w2v, y_train)
y_pred_rf = rf_model.predict(X_test_w2v)
print("\nWord2Vec + RandomForest Report:")
print(classification_report(y_test, y_pred_rf, target_names=target_names))